# 03 - Metaheuristicas (PSO) - Telemetry Heart AI

Ejecuta el optimizador **PSO** de produccion (`PSOOptimizer`) sobre el dataset
de triaje, optimizando 7 pesos clinicos + 2 umbrales. Usa exactamente la misma
carga/normalizacion que el runtime (`app.tools.pso_tools._load_dataset`).

> El notebook original importaba `app.services.genetic_engine.GeneticEngine` y
> `app.services.pso_engine.PSOEngine`. Tras la refactorizacion el **Algoritmo
> Genetico fue retirado del codigo** y el PSO vive ahora en
> `app.services.optimizers.pso.PSOOptimizer`. Por eso aqui solo se corre PSO.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
import os
from pathlib import Path
# Ejecutar siempre desde la raiz del microservicio.
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print('cwd:', Path.cwd())

In [ ]:
from app.tools.pso_tools import _load_dataset
from app.services.optimizers.pso import PSOOptimizer, FEATURE_NAMES

X, y = _load_dataset(Path('app/data/synthetic_cases.csv'))
print('X:', X.shape, '| clases y:', np.bincount(y))

In [ ]:
print('Ejecutando PSO...')
optimizer = PSOOptimizer(n_particles=30, max_iter=30)
result = optimizer.optimize(X, y)
print('Mejor fitness:', round(result.best_fitness, 4))
print('Pesos:', [round(w, 3) for w in result.weights])
print('Umbrales:', [round(t, 3) for t in result.thresholds])
print('Metricas:', {k: round(v, 4) for k, v in result.metrics.items() if isinstance(v, (int, float))})

In [ ]:
# Curva de convergencia
curve = result.convergence_curve
plt.figure(figsize=(9, 5))
plt.plot(range(1, len(curve) + 1), curve, 'r-o', ms=3)
plt.title('Convergencia PSO')
plt.xlabel('Iteracion'); plt.ylabel('Mejor fitness (gbest, a minimizar)')
plt.grid(alpha=0.3)
Path('app/data/charts').mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig('app/data/charts/convergence.png', dpi=120)
plt.show()

In [ ]:
# Pesos clinicos optimizados por feature
plt.figure(figsize=(10, 5))
plt.bar(FEATURE_NAMES, result.weights, color='#1f77b4')
plt.title('Pesos clinicos optimizados (PSO)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('app/data/charts/pso_weights.png', dpi=120)
plt.show()
print('Metaheuristicas completas')